# Check exported Hugging Face dataset

Use this notebook after running the pipeline to inspect the saved `hf_dataset` folder.

In [3]:
from pathlib import Path

import pandas as pd
from datasets import DatasetDict, load_from_disk

# Update this path if your dataset lives somewhere else.
DATASET_DIR = Path("../data/processed/hf_dataset")
# Example for a single processed video:
# DATASET_DIR = Path("../data/processed/VIDEO_ID/hf_dataset")

print(f"Checking: {DATASET_DIR.resolve()}")
print("Exists:", DATASET_DIR.exists())

c:\Users\nextt\Desktop\yaswanth_v\capstone-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking: C:\Users\nextt\Desktop\yaswanth_v\capstone-project\data\processed\hf_dataset
Exists: True


In [4]:
dataset = load_from_disk(str(DATASET_DIR))
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'text', 'duration_s', 'video_id', 'start', 'end'],
        num_rows: 257
    })
    validation: Dataset({
        features: ['audio', 'sentence', 'text', 'duration_s', 'video_id', 'start', 'end'],
        num_rows: 28
    })
})

In [5]:
if isinstance(dataset, DatasetDict):
    split_names = list(dataset.keys())
else:
    split_names = ["train"]
    dataset = DatasetDict({"train": dataset})

summary_rows = []
for split_name in split_names:
    split_ds = dataset[split_name]
    summary_rows.append(
        {
            "split": split_name,
            "rows": len(split_ds),
            "columns": ", ".join(split_ds.column_names),
            "features": str(split_ds.features),
        }
    )

pd.DataFrame(summary_rows)

,split,rows,columns,features
0,train,257,"audio, sentence, text, duration_s, video_id, s...","{'audio': Audio(sampling_rate=16000, decode=Tr..."
1,validation,28,"audio, sentence, text, duration_s, video_id, s...","{'audio': Audio(sampling_rate=16000, decode=Tr..."


In [6]:
preview_split = split_names[0]
preview_columns = [
    col for col in ["text", "sentence", "duration_s", "video_id", "start", "end"]
    if col in dataset[preview_split].column_names
]

dataset[preview_split].select(range(min(5, len(dataset[preview_split])))).to_pandas()[preview_columns]

,text,sentence,duration_s,video_id,start,end
0,varaku unna [news] ayitē ivi [news] lōki,varaku unna [news] ayitē ivi [news] lōki,1.67,ve8XqZ3bRcM,00:00:07.680,00:00:09.350
1,[display] mottaṁ mīku [flat] kanipistadi.,[display] mottaṁ mīku [flat] kanipistadi.,2.31,ve8XqZ3bRcM,00:02:45.760,00:02:48.070
2,mana chānal ni sabskraib cēsukoni phālō,mana chānal ni sabskraib cēsukoni phālō,1.27,ve8XqZ3bRcM,00:09:03.120,00:09:04.389
3,[next] [news] lōki veḷḷē muṁdu,[next] [news] lōki veḷḷē muṁdu,0.87,ve8XqZ3bRcM,00:06:39.039,00:06:39.909
4,[refrigerator] [double] [door] [refrigerator],[refrigerator] [double] [door] [refrigerator],1.75,ve8XqZ3bRcM,00:01:46.399,00:01:48.149


In [7]:
dataset = dataset.with_format("torch")

sample = dataset[preview_split][0]

print("Available keys:", list(sample.keys()))
print("\nText:", sample.get("text"))
print("Sentence:", sample.get("sentence"))
print("Duration:", sample.get("duration_s"))
print("Video ID:", sample.get("video_id"))
print("Start:", sample.get("start"))
print("End:", sample.get("end"))

sample["audio"]

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

In [8]:
sample = dataset[preview_split][0]
print(type(sample["audio"]))
print(sample["audio"])

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

In [9]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(decode=False))

In [10]:
sample = dataset[preview_split][0]

print(sample.keys())
print(sample["text"])
print(sample["sentence"])
print(sample["duration_s"])
print(sample["video_id"])
print(sample["start"])
print(sample["end"])

print(sample["audio"])   # SAFE (no crash)

dict_keys(['audio', 'sentence', 'text', 'duration_s', 'video_id', 'start', 'end'])
varaku unna [news] ayitē ivi [news] lōki
varaku unna [news] ayitē ivi [news] lōki
tensor(1.6700)
ve8XqZ3bRcM
00:00:07.680
00:00:09.350
{'bytes': b'RIFF\xe4\xd0\x00\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\xc0\xd0\x00\x00\xb4\xf8\xc8\xf7k\xf7\x7f\xf7\xfc\xf7\xec\xf8#\xfam\xfb\x9b\xfc}\xfd\x02\xfe\x1f\xfe\xc6\xfd\x1d\xfd*\xfc\xd6\xfa9\xf9t\xf7\xc8\xf5W\xf4B\xf3\xab\xf2\xaa\xf2F\xf3k\xf4\xf4\xf5\xc4\xf7\x8b\xf9\x1d\xfbM\xfc\xfc\xfc:\xfd\x16\xfd\xb2\xfc<\xfc\xf9\xfb\xf9\xfb:\xfc\xc9\xfc\x90\xfdr\xfe1\xff\xb7\xff\xf3\xff\xd1\xff[\xff\xa4\xfe\xc8\xfd\xf9\xfcK\xfc\xd2\xfb\xc0\xfb9\xfc.\xfdc\xfeo\xffg\x00y\x01\x9a\x02\xad\x03q\x04\x13\x05\xad\x05K\x06\xfd\x06\xa7\x07\'\x08v\x08\xa5\x08\xf2\x08[\t\xb5\t\xe2\t\x08\n[\n\xf8\n\xca\x0bx\x0c\xfd\x0cX\r\xa6\r\xe8\r\xcf\r9\r-\x0c\xf0\n\xe8\t%\t\x92\x08\x1e\x08\xde\x07\xf2\x07>\x08\x87\x08\x84\x08\x1a\x08]\x07p\x06]\x05#\x